# 12 v2 — Final test protokolü / selective answering kilidi

**Amaç:** Stage 12 v1'de tanımlanan 400 soruluk hedefi, **final test inference / test sonuçları incelenmeden önce**, 250 soruluk sabit bağımsız teste uyarlamak; 300 aday hedefini ve selective answering ölçümlerini kesinleştirmek.

Bu notebook **model eğitmez, yeni soru üretmez, test etiketlerini açmaz, final test skoru hesaplamaz**. Stage 08, 10, 11'in freeze edilmiş SHA bağlantılarını doğrular. Mevcut Stage 12 v1 dosyalarına dokunmaz; varsa v1 manifestini doğrular ve v2'ye bağlar, yoksa v2'yi kendi başına dondurur.

Çıktılar: `results/stage12/validated_v007/final_test_protocol_v2.json` ve `stage12_protocol_manifest_v2.json`.

**Ön koşul:** Stage 11'in en son immutable çıktılar/manifest hücresi Colab'da çalışmış olmalıdır. Stage 12 v1 eksik kaydedilmişse (yalnız protocol veya yalnız manifest) notebook hata verir; bu durumda önce v1 durumu açıklığa kavuşturulmalıdır.


## 1 — Drive, yardımcılar ve frozen artifact path’leri

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, os
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
VERSION = 'validated_v007'
ASSEMBLY = 'feature_assembly_v2_repair1'
BASE = ROOT / 'features' / 'assembled' / VERSION
M08 = BASE / 'stage08_manifest_v2_repair1.json'
SCHEMA = BASE / 'feature_schema_v1.json'
MATRIX = BASE / 'extended_model_matrix_v2_repair1.parquet'
REPAIR = ROOT / 'results' / 'stage05' / f'{VERSION}_stage05_self_confidence_repair_v1.json'

def sha_file(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p,encoding='utf-8') as f:
        return json.load(f)

def require(p):
    if not Path(p).is_file():
        raise FileNotFoundError(str(p))

def freeze_json(path, data):
    blob=(json.dumps(data,ensure_ascii=False,indent=2,sort_keys=True)+'\n').encode('utf-8')
    if path.exists():
        if path.read_bytes()!=blob:
            raise RuntimeError(f'IMMUTABLE PROTOCOL CONFLICT: {path}')
        print('Existing frozen artifact verified:',path)
        return
    tmp=path.with_name(path.name+'.tmp')
    with open(tmp,'xb') as f:
        f.write(blob)
    os.replace(tmp,path)
    print('Frozen artifact created:',path)

Mounted at /content/drive


## 2 — Stage 08 / 10 / 11 SHA ve veri kapsamı doğrulaması

In [2]:
for p in [M08, SCHEMA, MATRIX, REPAIR]:require(p)
m08,s,repair=read_json(M08),read_json(SCHEMA),read_json(REPAIR)
if m08['dataset_version'] != VERSION or m08['assembly_version'] != ASSEMBLY:raise RuntimeError('Stage08 version mismatch')
if s['dataset_version'] != VERSION or s['assembly_version'] != ASSEMBLY:raise RuntimeError('Schema version mismatch')
if sha_file(SCHEMA)!=m08['outputs']['feature_schema_file_sha256']:raise RuntimeError('Schema SHA mismatch')
if sha_file(MATRIX)!=m08['outputs']['extended_model_matrix_sha256']:raise RuntimeError('Matrix SHA mismatch')
if sha_file(REPAIR)!=m08['upstream']['stage05_self_confidence_repair_manifest_sha256']:raise RuntimeError('Repair SHA mismatch')
if s['schema_sha256']!=m08['feature_schema_sha256']:raise RuntimeError('Schema content SHA mismatch')
repair_sidecar=Path(repair['repair_file']);require(repair_sidecar)
if sha_file(repair_sidecar)!=repair['repair_file_sha256']:raise RuntimeError('Repair sidecar SHA mismatch')

D10=ROOT/'results'/'stage10'/VERSION/('stage08_'+sha_file(M08)[:12]+'_repair1')
M10=D10/'stage10_dev_manifest_v1.json';require(M10)
m10=read_json(M10)
if m10['created_from_stage08_sha256']!=sha_file(M08):raise RuntimeError('Stage10 -> Stage08 mismatch')
for name,entry in m10['outputs'].items():
    p=Path(entry['path']);require(p)
    if sha_file(p)!=entry['sha256']:raise RuntimeError(f'Stage10 {name} SHA mismatch')
contract10_path=Path(m10['outputs']['contract']['path'])
c10=read_json(contract10_path)
if c10['stage05_repair_manifest_sha256']!=sha_file(REPAIR):raise RuntimeError('Stage10 repair linkage mismatch')
if c10['scope']!='CLEAN_DEV: exclude_from_final_test==0':raise RuntimeError('Stage10 scope mismatch')

D11=ROOT/'results'/'stage11'/VERSION/('stage10_'+sha_file(M10)[:12]+'_xgb_nested_v1')
M11=D11/'stage11_dev_manifest_v1.json';require(M11)
m11=read_json(M11)
if m11['stage10_manifest_sha256']!=sha_file(M10):raise RuntimeError('Stage11 -> Stage10 mismatch')
if m11['stage08_manifest_sha256']!=sha_file(M08):raise RuntimeError('Stage11 -> Stage08 mismatch')
if m11['stage05_repair_manifest_sha256']!=sha_file(REPAIR):raise RuntimeError('Stage11 repair linkage mismatch')
for name,entry in m11['output_files'].items():
    p=Path(entry['path']);require(p)
    if sha_file(p)!=entry['sha256']:raise RuntimeError(f'Stage11 {name} SHA mismatch')
contract11_path=Path(m11['output_files']['contract']['path']);c11=read_json(contract11_path)
if c11['stage10_manifest_sha256']!=sha_file(M10):raise RuntimeError('Stage11 contract -> Stage10 mismatch')
if c11['stage08_manifest_sha256']!=sha_file(M08):raise RuntimeError('Stage11 contract -> Stage08 mismatch')
if c11['primary_metric']!='Brier':raise RuntimeError('Stage11 Brier contract mismatch')

full=pd.read_parquet(MATRIX)
core=list(s['core_features'])
ycol=str(s['target'])
clean=full.loc[full['exclude_from_final_test'].astype(int)==0].copy()
if not (len(full)==3000 and len(clean)==2910 and len(core)==8 and clean.groupby('model_id').size().eq(970).all()):
    raise RuntimeError('Frozen development data shape mismatch')
if clean.duplicated(['model_id','question_id']).any():raise RuntimeError('Development duplicate keys')
if c10['logistic_feature_contract']['core']!=core:raise RuntimeError('Logistic CORE mismatch')
if c11['core_features']!=core:raise RuntimeError('XGB CORE mismatch')
if c11['xgb_methods'][0]!='xgb_core_nested':raise RuntimeError('XGB primary method mismatch')
if c10['n_splits']!=5 or c10['seeds']!=[1101,2202,3303]:raise RuntimeError('CV contract mismatch')
print('STAGE 08/10/11 PROVENANCE + SHA: PASS')
print('Development:',len(clean),'model-answer records; 970 questions/model')
print('CORE features:',core)
print('Stage11:',M11)

STAGE 08/10/11 PROVENANCE + SHA: PASS
Development: 2910 model-answer records; 970 questions/model
CORE features: ['mean_logprob', 'min_logprob', 'mean_token_entropy_normalized', 'output_content_token_count', 'self_confidence_scaled', 'verification_p_correct', 'self_consistency', 'unknown_response']
Stage11: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage11/validated_v007/stage10_ada008b24140_xgb_nested_v1/stage11_dev_manifest_v1.json


## 1 — Kesinleştirilmiş yöntem ve test sözleşmesi

- Ana karşılaştırma: **model başına XGBoost CORE – Logistic CORE Brier farkı**, aynı **sekiz** frozen CORE feature, üç ayrı calibrator. `970 CLEAN_DEV` soru/model ile eğitim. Önceden kullanılan 30 model-selection sorusu eğitim dışında kalır.
- Logistic: Stage 10'daki `StandardScaler` + `LogisticRegression` sözleşmesi; scaler yalnız development verisinde fit.
- XGBoost: Stage 11'in dondurulmuş dört adaylı grid'i (`max_depth∈{1,2}`, `n_estimators∈{60,120}`), shared params ve Brier minimizasyonu. Final training'de **970 CLEAN_DEV üzerinde tek 3-fold inner CV** (seed=17011), seçilen konfigürasyonla 970 kayıt üzerinde refit. Nihai test etiketleri bu süreçte kullanılmaz.
- Ana test: **250 yeni benzersiz Türkçe olgusal soru × 3 model = 750 model–cevap**. **300 aday hedefi** başlangıç hedefidir; kalite/duplicate sonrası 250 geçerli kayıt kalmazsa inference'dan *önce* ek aday üret; hiçbir zaman geçersiz soruyu kotayı doldurmak için alma.
- Cevap anahtarı güvenilir referanslarla **model inference öncesi** doğrulanır; `reference_answer`, alias ve gold bilgisi ana cevap ya da confidence prompt'una girmez. `validated_v007` içindeki 1000 soruyla **exact + semantik** tekrar kontrolü; yeni test soruları arasında da tekrar kontrolü.
- Soru kaydı, kategori/answer type hedefleri, gerekirse önceden tanımlı seçim seed'i ve Stage 04/05/06 frozen prompt/feature sürümleri Stage 13'te inference'dan önce dondurulacak. Model doğruluğuna bakarak soru eleme/ekleme yapılmaz.
- Ana metrik: Brier; ikincil: NLL, ECE (10 eşit genişlikte bin), AUROC, AUPRC, **AURC**, risk@coverage (20/50/80/100%), threshold@confidence (0.80, 0.90) için empirical risk ve gerçek coverage. Baseline: ham verification ve **yalnız development etiketlerinden** hesaplanan prevalence.
- Önceden belirlenmiş seçicilik politikası: skora göre azalan sırada sıralayıp tam coverage için `k=ceil(c*N)` cevabı seç (eşit skorda `question_id` artan); `AURC = (1/N)∑_{k=1}^{N} Risk@k`. Böylece AURC hesaplaması final sonuçlar görülmeden tanımlanmıştır. Confidence threshold'ları yalnız olasılık tahmini olarak değerlendirilen Logistic/XGB için; herhangi bir threshold hiç cevap seçmezse risk `NaN`, coverage `0` kaydedilir.
- Aynı soru üzerindeki yöntem farkları için model başına soru düzeyinde paired bootstrap, B=4000, seed=20260921. Birincil Brier farkı için iki yönlü %95 percentile aralığı; secondary karşılaştırmalar keşifsel. Testte 250 bağımsız soru vardır, 750 bağımsız soru yoktur.
- Final test ana cevapları/özellikleri ve calibrator tahminleri **doğruluk etiketleri açılmadan önce** SHA ile dondurulur. Test etiketleri hiperparametre, feature, model veya eşik seçimine girmez. Yeni yöntemler gelecekte ayrı bir development sürümü ve *yeni* test kohortu ile incelenebilir.

**Önemli:** Stage 12 v2 yalnız test hedefi ve değerlendirme kapsamı için **ön-test gerekçeli ek protokoldür**. Stage 12 v1 dosyaları varsa aynı haliyle tutulur. Önceki test metrikleri incelenmediği kullanıcı tarafından teyit edilen bir prosedür varsayımıdır; notebook başka konumdaki tüm test verilerini tarayarak bunu kanıtlayamaz.


## 3 — 1000 soruluk `validated_v007` dağılımından 250 test sorusu için kotalar

In [3]:
# Kaynak: dondurulmuş 1000 soruluk validated_v007 kayıtları.
# Stage 13 bu kotaları yalnız model inference ÖNCESİNDE kullanacak.
from collections import Counter
VALIDATED = ROOT / 'data' / 'validated' / (VERSION + '.json')
require(VALIDATED)
dataset = read_json(VALIDATED)
assert len(dataset) == 1000
ids = [str(r['question_id']) for r in dataset]
assert len(set(ids)) == 1000
cat_counts = Counter(str(r['category']) for r in dataset)
type_counts = Counter(str(r['answer_type']) for r in dataset)
assert len(cat_counts) == 13

# Kategoriler development verisinde zaten yaklaşık dengeli.
# 13*19 + 3 = 250: en kalabalık üç kategori 20, diğerleri 19.
category_quotas = {cat: 19 for cat in cat_counts}
for cat in sorted(cat_counts, key=lambda c: (-cat_counts[c], c))[:3]:
    category_quotas[cat] += 1

# Önceki mesajda önceden belirlenen, 1000 soruluk answer-type
# dağılımına yaklaşık orantılı sabit hedefler.
answer_type_quotas = {
    'ENTITY_PERSON': 43,
    'ENTITY_LOCATION': 42,
    'ENTITY_ORGANIZATION': 25,
    'ENTITY_OTHER': 25,
    'DATE': 33,
    'NUMBER': 25,
    'SHORT_PHRASE': 25,
    'NUMBER_WITH_UNIT': 17,
    'BINARY': 15,
}
assert set(answer_type_quotas) == set(type_counts)
assert sum(category_quotas.values()) == 250
assert sum(answer_type_quotas.values()) == 250

print('VALIDATED_V007 DISTRIBUTIONS: PASS')
print('Category targets:')
for k in sorted(category_quotas):
    print(f'  {k:16s}: dev={cat_counts[k]:3d}, final={category_quotas[k]:2d}')
print('Answer-type targets:')
for k in sorted(answer_type_quotas):
    print(f'  {k:20s}: dev={type_counts[k]:3d}, final={answer_type_quotas[k]:2d}')
print('SUM:', sum(category_quotas.values()), sum(answer_type_quotas.values()))


VALIDATED_V007 DISTRIBUTIONS: PASS
Category targets:
  Bilim           : dev= 80, final=20
  Coğrafya        : dev= 76, final=19
  Dil             : dev= 79, final=20
  Edebiyat        : dev= 78, final=19
  Ekonomi         : dev= 74, final=19
  Felsefe         : dev= 73, final=19
  Genel Kültür    : dev= 75, final=19
  Matematik       : dev= 75, final=19
  Sanat           : dev= 79, final=20
  Spor            : dev= 78, final=19
  Tarih           : dev= 78, final=19
  Teknoloji       : dev= 78, final=19
  Uzay            : dev= 77, final=19
Answer-type targets:
  BINARY              : dev= 61, final=15
  DATE                : dev=130, final=33
  ENTITY_LOCATION     : dev=170, final=42
  ENTITY_ORGANIZATION : dev=100, final=25
  ENTITY_OTHER        : dev=100, final=25
  ENTITY_PERSON       : dev=170, final=43
  NUMBER              : dev=100, final=25
  NUMBER_WITH_UNIT    : dev= 69, final=17
  SHORT_PHRASE        : dev=100, final=25
SUM: 250 250


## 4 — V1 dosyaları mevcutsa kontrol et; hiçbir v1 artifact’ına yazma

In [4]:
# Eski protokol varsa değiştirme: iki dosya birlikte bulunmalı.
OUT = ROOT / 'results' / 'stage12' / VERSION
V1_PROTOCOL = OUT / 'final_test_protocol_v1.json'
V1_MANIFEST = OUT / 'stage12_protocol_manifest_v1.json'
if V1_PROTOCOL.exists() != V1_MANIFEST.exists():
    raise RuntimeError('Stage12 v1 partially frozen: inspect v1 protocol + manifest first')
previous = None
if V1_PROTOCOL.exists():
    v1, vm = read_json(V1_PROTOCOL), read_json(V1_MANIFEST)
    if vm['protocol_sha256'] != sha_file(V1_PROTOCOL):
        raise RuntimeError('Stage12 v1 protocol SHA mismatch')
    if v1.get('protocol_version') != 'final_test_protocol_v1':
        raise RuntimeError('Unexpected Stage12 v1 version')
    if v1['stage11_manifest_sha256'] != sha_file(M11):
        raise RuntimeError('Stage12 v1 / frozen Stage11 mismatch')
    if v1['test_registration']['new_questions_target'] != 400:
        raise RuntimeError('Unexpected Stage12 v1 sample size')
    previous = {
        'version': 'final_test_protocol_v1',
        'protocol_file': str(V1_PROTOCOL),
        'protocol_sha256': sha_file(V1_PROTOCOL),
        'manifest_file': str(V1_MANIFEST),
        'manifest_sha256': sha_file(V1_MANIFEST),
    }
    print('Stage12 v1 located & SHA verified (left untouched)')
else:
    print('Stage12 v1 files not present: standalone pre-test v2 will be frozen')


Stage12 v1 located & SHA verified (left untouched)


## 5 — Machine-readable v2 protokolünü immutable olarak kaydet

In [5]:
# Bu hücre yalnızca sonuçlar görülmeden tanımlanmış protokolü kaydeder.
# Aynı isimli v2 dosyası varsa farklı içerikle ÜZERİNE YAZMAZ.
OUT.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = OUT / 'final_test_protocol_v2.json'
MANIFEST_PATH = OUT / 'stage12_protocol_manifest_v2.json'

models = sorted(clean['model_id'].unique().tolist())
if models != sorted(['Qwen/Qwen3-4B', 'google/gemma-3-4b-it', 'microsoft/Phi-4-mini-instruct']):
    raise RuntimeError('Unexpected frozen model IDs')
if sorted(cat_counts) != sorted(category_quotas):
    raise RuntimeError('Category mismatch')
if set(type_counts) != set(answer_type_quotas):
    raise RuntimeError('Answer-type mismatch')

protocol = {
    'stage': '12_final_test_protocol_lock',
    'protocol_version': 'final_test_protocol_v2',
    'amendment_reason': 'Pre-test reduction of final n from 400 to 250 to control annotation/inference workload; prospective selective answering/AURC definitions added; no final test metrics inspected for this change.',
    'supersedes_if_present': previous,
    'development_dataset_version': VERSION,
    'validated_dataset_sha256': sha_file(VALIDATED),
    'stage08_assembly_version': ASSEMBLY,
    'stage08_manifest_sha256': sha_file(M08),
    'stage08_matrix_sha256': sha_file(MATRIX),
    'stage08_schema_sha256': sha_file(SCHEMA),
    'stage05_repair_manifest_sha256': sha_file(REPAIR),
    'stage10_manifest_sha256': sha_file(M10),
    'stage11_manifest_sha256': sha_file(M11),
    'model_ids': models,
    'train_scope': 'CLEAN_DEV; exclude_from_final_test==0; 30 stage03 model-selection questions excluded',
    'n_training_questions_per_model': 970,
    'feature_names_ordered': core,
    'target_column': ycol,
    'final_test_registry': {
        'n_independent_questions': 250,
        'n_candidate_questions_initial_target': 300,
        'candidate_refill_rule': 'If fewer than 250 eligible candidates pass checks, produce additional candidates before any target-LLM inference; never lower validity rules or select using target-model results.',
        'expected_model_answer_rows': 750,
        'category_development_counts': dict(sorted(cat_counts.items())),
        'category_final_quotas': dict(sorted(category_quotas.items())),
        'answer_type_development_counts': dict(sorted(type_counts.items())),
        'answer_type_final_quotas': dict(sorted(answer_type_quotas.items())),
        'difficulty_preinference_target_counts': {'easy':75,'medium':125,'hard':50},
        'difficulty_assigned_without_target_model_outputs': True,
        'final_candidate_selection_seed': 20260921,
        'preinference_freeze_question_id_question_category_answer_type_gold_aliases_and_references': True,
        'gold_answer_never_in_target_model_prompts_or_confidence_features': True,
        'candidate_quality_checks': [
            'single_unambiguous_factual_answer',
            'fact_checked_using_explicit_external_source_and_aliases',
            'time_stable_or_explicit_time_scope',
            'exact_and_semantic_dedup_against_all_1000_validated_v007_questions',
            'exact_and_semantic_dedup_within_new_test_questions',
            'no_target_model_correctness_based_selection',
        ],
        'required_gold_reference_fields': ['source_name','source_reference'],
        'question_registry_created_in_stage13': True,
    },
    'primary_comparison': {
        'reference_method': 'logistic_core',
        'alternative_method': 'xgb_core',
        'primary_metric': 'brier',
        'effect': 'brier_xgb_minus_brier_logistic',
        'paired_unit': 'question_id; each LLM evaluated separately',
        'one_sided_selection_after_test': False,
    },
    'logistic_core': {
        'features': core,
        'estimator': 'sklearn.pipeline(StandardScaler,LogisticRegression)',
        'scaler_fit': 'development_only',
        'params': {'C':float(c10['logistic_C']), 'solver':'lbfgs', 'max_iter':3000, 'random_state':17},
        'training_records_per_llm': 970,
    },
    'xgb_core': {
        'features': core,
        'estimator': 'xgboost.XGBClassifier',
        'shared_parameters': c11['xgb_shared_parameters'],
        'hyperparameter_candidates': c11['xgb_hyperparameter_candidates'],
        'hyperparameter_selection': {
            'data':'970 CLEAN_DEV training records ONLY per model',
            'inner_cv':'StratifiedKFold(n_splits=3, shuffle=True, random_state=17011)',
            'selection_metric':'mean_validation_brier',
            'tie_break':'mean_Brier,n_estimators,max_depth',
            'final_refit':'selected parameters, all 970 CLEAN_DEV records per model',
        },
    },
    'prespecified_baselines': {
        'raw_verification':'repair-aware raw verification_p_correct; no final-label fitting',
        'train_prevalence':'mean development manual_is_correct per model, constant on new test; do not use test prevalence to fit',
    },
    'evaluation': {
        'primary_brier': {'lower_is_better':True,'paired_diff':'xgb_minus_logistic'},
        'secondary_nll': {'clip_eps':1e-6},
        'secondary_ece': {'n_bins':10,'bin_width':'equal_[0,1]'},
        'secondary_auroc': {'undefined_if_only_one_target_class':True},
        'secondary_auprc': {'positive_class':1,'report_test_prevalence':True},
        'selective_answering': {
            'rank':'descending_confidence_then_question_id_ascending',
            'coverage_policy':'select top k, k=max(1,ceil(coverage*N))',
            'coverage_levels':[0.2,0.5,0.8,1.0],
            'empirical_risk':'number_incorrect_selected / number_selected',
            'aurc':'discrete mean of risk@k for k=1..N, i.e. sum(risk@k)/N',
            'fixed_probability_thresholds':[0.8,0.9],
            'threshold_policy':'select p_correct >= threshold for logistic_core and xgb_core; report attained coverage, n_selected, empirical risk; n_selected=0 -> risk=null, not 0',
            'confidence_thresholds_for_raw_verification':False,
            'no_final_test_tuned_thresholds':True,
        },
    },
    'uncertainty': {
        'paired_bootstrap_resampling_unit':'question_id within each LLM',
        'bootstrap_repetitions':4000,
        'bootstrap_seed':20260921,
        'primary_brier_interval':'two_sided_percentile_95',
        'coverage_risk_and_AURC_intervals':'exploratory question-level paired bootstrap, same prespecified seed/repetitions',
        'no_independent_750_question_claim':True,
    },
    'annotation': {
        'gold_and_aliases_verified_before_model_inference':True,
        'manual_label':'manual_is_correct 0 or 1',
        'labeler_blind_to_confidence_and_calibrator_scores':True,
        'reference_and_gold_not_predictors':True,
        'adjudication_policy_to_be_frozen_before_labeling':True,
    },
    'new_test_format_repair': {
        'only_format_or_parse_failure':True,
        'preserve_original_raw_attempts_and_SHA':True,
        'retry_prompt_and_max_attempts_frozen_in_stage13_before_inference':True,
        'repair_is_new_inference_not_just_reparsing':True,
        'no_posthoc_selection_by_correctness':True,
    },
    'evaluation_gates': {
        'stage13_question_ids_gold_sources_SHA_locked_preinference':True,
        'main_answers_features_and_final_confidence_predictions_SHA_locked_before_opening_test_manual_labels':True,
        'no_test_based_retraining_feature_picking_or_threshold_picking':True,
        'future_new_methods_require_separate_development_version_and_unseen_test':True,
    },
    'status':'LOCKED_PROTOCOL_NO_FINAL_TEST_EVALUATION',
}
freeze_json(PROTOCOL_PATH, protocol)
manifest = {
    'stage':'12_final_test_protocol_lock',
    'protocol_version':'final_test_protocol_v2',
    'protocol_file':str(PROTOCOL_PATH),
    'protocol_sha256':sha_file(PROTOCOL_PATH),
    'supersedes_v1_if_present':previous,
    'upstream': {
        'validated_v007_file':str(VALIDATED),
        'validated_v007_sha256':sha_file(VALIDATED),
        'stage08_manifest':str(M08),
        'stage08_manifest_sha256':sha_file(M08),
        'stage10_manifest':str(M10),
        'stage10_manifest_sha256':sha_file(M10),
        'stage11_manifest':str(M11),
        'stage11_manifest_sha256':sha_file(M11),
    },
    'stage13_question_registry_created_by_this_notebook':False,
    'test_inference_executed_by_this_notebook':False,
    'test_labels_read_by_this_notebook':False,
    'status':'STAGE12_V2_PROTOCOL_FROZEN',
}
freeze_json(MANIFEST_PATH,manifest)
if read_json(MANIFEST_PATH)['protocol_sha256'] != sha_file(PROTOCOL_PATH):
    raise RuntimeError('Stage12 v2 protocol / manifest SHA mismatch')
print('STAGE 12 V2 PROTOCOL: FROZEN')
print('Protocol:',PROTOCOL_PATH)
print('Manifest:',MANIFEST_PATH)
print('Final test size:',protocol['final_test_registry']['n_independent_questions'])
print('Candidate target:',protocol['final_test_registry']['n_candidate_questions_initial_target'])
print('Selective coverage:',protocol['evaluation']['selective_answering']['coverage_levels'])
print('Selective p-thresholds:',protocol['evaluation']['selective_answering']['fixed_probability_thresholds'])
print('Final test inference in this notebook: NO')


Frozen artifact created: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage12/validated_v007/final_test_protocol_v2.json
Frozen artifact created: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage12/validated_v007/stage12_protocol_manifest_v2.json
STAGE 12 V2 PROTOCOL: FROZEN
Protocol: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage12/validated_v007/final_test_protocol_v2.json
Manifest: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage12/validated_v007/stage12_protocol_manifest_v2.json
Final test size: 250
Candidate target: 300
Selective coverage: [0.2, 0.5, 0.8, 1.0]
Selective p-thresholds: [0.8, 0.9]
Final test inference in this notebook: NO


## 5 — Stage 13 başlamadan önce kontrol listesi

1. `STAGE 08/10/11 PROVENANCE + SHA: PASS`, `VALIDATED_V007 DISTRIBUTIONS: PASS` ve `STAGE 12 V2 PROTOCOL: FROZEN` çıktısını doğrula. v1 varsa değiştirilmediğini kontrol et.
2. Stage 13 için 300 aday üret; kategori, cevap türü, kaynak ve gold answer'ı **hedef model cevaplarını görmeden** doğrula. Geçerli adaylardan kotaları ve zorluk hedeflerini sağlayan 250 benzersiz soru seç; gerekirse önceden aday sayısını artır.
3. `final_test_questions_v1.json` (yalnız inference alanları) ve ayrı `final_test_gold_v1.json` (gold, aliases, source) üret; iki dosyanın SHA değerlerini, tüm ID'leri ve eski 1000 soru ile tekrar denetimini Stage 13 manifestinde kilitle.
4. Ardından frozen Stage 04/05/06 inference akışı, confidence sinyalleri, dev-only final calibrator fit, **etiketleri açmadan önce test tahminlerini dondurma**, kör etiketleme ve yalnız bir kez final değerlendirme.

Test sonuçlarından sonra ek boosting yöntemi veya yeni veriyle calibrator geliştirmek istersen **yeni veri sürümü ve yeni bağımsız test kohortu** aç; bu final test ile yeniden yöntem seçimi yapma.
